# Risk labels on our forecasts

Applies the risk flags of [`risk-definition.ipynb`](../03_risk_classification/risk-definition.ipynb) (thresholds, day rules, limits of the label) to the test-window forecasts of spec 09's picks ([`visualization-01-regression-best-models.ipynb`](visualization-01-regression-best-models.ipynb)) and of SMARD, and scores them against the actual labels; forecast method and caveats are in [`regression-models-claude.ipynb`](regression-models-claude.ipynb).
Reads `data/risk_classification/risk_labels_{daily,hourly}.csv`, `data/models/model_{forecast_errors_hourly,scoreboard}.csv` and `data/metrics/smard_forecast_errors_hourly.csv`; writes `data/risk_classification/model_risk_labels_{daily,hourly}.csv`.
*The label is a national-balance proxy, not a validated intervention record.*

## 1 Setup and checks

### 1.1 Settings

In [ ]:
from pathlib import Path

import holidays
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch, Rectangle
from matplotlib.transforms import ScaledTranslation

PICKS = {"high": "xgb_hybrid", "low": "random_forest_hybrid"}  # spec 09 rank 1: high_extreme / low_extreme
PICK_CATEGORIES = {"high": ["high_extreme"], "low": ["low_extreme", "below_zero"]}  # checked in 1.3
SPLIT = "rolling"            # split method of the picks (spec 09's CANDIDATE_SPLIT)
BASES = {"high": ["rolling"], "low": ["rolling", "zero"]}
RULES = ["any", "3h"]
MIN_BIN_SHARE = 0.5          # spec 09's forecast-share floor, used by the pick check
ZOOM_WEEKS = {"high": [None, None], "low": [None, None]}  # a Monday "YYYY-MM-DD" overrides a zoom rule
EXPORT_ENABLED = True
MARGIN_WINDOW = 10_000       # MWh: axis range of the day-margin plot around 0

# Day rules as durations, as in risk-definition.ipynb §3.1 (RESOLUTION is measured in §2.1)
PERSISTENCE = pd.Timedelta("3h")
DAY_COMPLETENESS = 23 / 24

# Label and colour per row, copied from visualization-01-regression-best-models.ipynb
STYLE = {
    "sarimax_fourier": {"label": "SARIMAX + Fourier", "color": "#D9A53A"},
    "lgbm_direct": {"label": "LightGBM direct", "color": "#2C6EBA"},
    "lgbm_hybrid": {"label": "LightGBM hybrid", "color": "#2F8F5B"},
    "xgb_direct": {"label": "XGBoost direct", "color": "#E95D0F"},
    "xgb_hybrid": {"label": "XGBoost hybrid", "color": "#B10F0F"},
    "linear_direct": {"label": "Ridge direct", "color": "#7A4FA3"},
    "random_forest_hybrid": {"label": "Random forest hybrid", "color": "#8C564B"},
    "actual": {"label": "Actual residual load", "color": "#1C1C1C"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A", "linestyle": "--"},
}

# Thresholds, flagged hours and hit cells per (direction, basis), never a model's colour
TAIL_COLOR = {
    ("high", "rolling"): "#E0436B",  # crimson-pink, apart from both XGBoost reds
    ("low", "rolling"): "#17BECF",   # cyan, low_extreme
    ("low", "zero"): "#9EDAE5",      # light cyan, below_zero
}

# Unflagged day and hour cells; flagged ones take the TAIL_COLOR (hit: fill, miss: outline, false alarm: hatch)
OUTCOME_COLOR = {
    "quiet": "#F2F4F7",
    "not evaluable": "white",  # also days outside the test window
}
HOLIDAY_COLOR = "#8A9A2B"  # olive, dashed: no model or tail colour


def style_timeseries(ax, title, ylabel):
    """Apply the EDA time-series chart style to the matplotlib axes.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, color="grey")

    # Light horizontal grid lines behind the data, no box on the top and right
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    # Tick labels without tick marks
    ax.tick_params(colors="black", length=0)

    # x axis: one label per year, minor ticks every quarter
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # y axis: thousands separator, no decimals (e.g. 45,000)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")


def style_plain(ax, title, ylabel, xlabel):
    """The same look for plots without a time axis."""
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel(xlabel, color="grey")
    ax.set_ylabel(ylabel, color="grey")
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors="black", length=0)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")

### 1.2 Loading

In [ ]:
# data/: first in the working directory, then in each parent
DATA_DIR = next(
    (folder / "data" for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(f"No data/ folder found in {Path.cwd()} or any parent folder.")

LABELS_SOURCE = "notebooks/03_risk_classification/risk-definition.ipynb"
MODEL_SOURCE = "notebooks/05_modeling/regression-models-claude.ipynb with EXPORT_ENABLED = True"
SMARD_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
SOURCES = {
    "labels_daily": ("risk_classification/risk_labels_daily.csv", LABELS_SOURCE),
    "labels_hourly": ("risk_classification/risk_labels_hourly.csv", LABELS_SOURCE),
    "hourly": ("models/model_forecast_errors_hourly.csv", MODEL_SOURCE),
    "scoreboard": ("models/model_scoreboard.csv", MODEL_SOURCE),
    "smard": ("metrics/smard_forecast_errors_hourly.csv", SMARD_SOURCE),
}
STAMP = "%Y-%m-%d %H:%M:%S"


def load(name):
    """One file as a DataFrame; stops if it is missing."""
    path, producer = SOURCES[name]
    path = DATA_DIR / path
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Create it by running {producer}.")
    frame = pd.read_csv(path)
    if "timestamp" in frame.columns:
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], format=STAMP)
    if "date" in frame.columns:
        frame["date"] = pd.to_datetime(frame["date"], format="%Y-%m-%d")
    return frame


# Label columns stay three-state: an empty cell is "not evaluable", never False
LABEL_PAIRS = [(direction, basis) for direction in BASES for basis in BASES[direction]]
DAILY_THRESHOLDS = [f"{d}_threshold_{b}" for d, b in LABEL_PAIRS]
DAILY_FLAGS = [f"{d}_risk_{b}_{r}" for d, b in LABEL_PAIRS for r in RULES]
DAILY_RANGES = [f"{d}_range_{e}_{b}_{r}" for d, b in LABEL_PAIRS for r in RULES for e in ("start", "end")]
HOURLY_FLAGS = [f"{d}_risk_hour_{b}" for d, b in LABEL_PAIRS]

labels_daily = load("labels_daily").set_index("date")[DAILY_THRESHOLDS + DAILY_FLAGS + DAILY_RANGES]
labels_daily[DAILY_FLAGS] = labels_daily[DAILY_FLAGS].astype("boolean")
for column in DAILY_RANGES:
    labels_daily[column] = pd.to_datetime(labels_daily[column], format=STAMP)
labels_hourly = load("labels_hourly").set_index("timestamp")[["residual_load"] + HOURLY_FLAGS]
labels_hourly[HOURLY_FLAGS] = labels_hourly[HOURLY_FLAGS].astype("boolean")

hourly = load("hourly")
scoreboard = load("scoreboard")
smard = load("smard").set_index("timestamp")

# Test window: the hours and local dates of the model export
TEST_HOURS = pd.DatetimeIndex(sorted(hourly["timestamp"].unique()))
TEST_DAYS = pd.DatetimeIndex(sorted(TEST_HOURS.normalize().unique()))

# The picks' forecasts; a missing pick stays an empty column and fails the pick check
picked = hourly[hourly["split_method"].eq(SPLIT) & hourly["model"].isin(PICKS.values())]
FORECAST = picked.pivot(index="timestamp", columns="model", values="forecast").reindex(
    index=TEST_HOURS, columns=list(dict.fromkeys(PICKS.values()))
)
ACTUAL = picked.groupby("timestamp")["residual_load"].first().reindex(TEST_HOURS)
SMARD_FORECAST = smard["fc_residual_load"].reindex(TEST_HOURS)

# Common hours: an actual, SMARD's forecast and a forecast from both picks
COMMON = TEST_HOURS[ACTUAL.notna() & SMARD_FORECAST.notna() & FORECAST.notna().all(axis=1)]

print(f"test window : {TEST_DAYS[0]:%Y-%m-%d} .. {TEST_DAYS[-1]:%Y-%m-%d} ({len(TEST_DAYS)} days, {len(TEST_HOURS):,} hours)")
print(f"picks       : high {PICKS['high']}, low {PICKS['low']} ({SPLIT} split)")
print(f"common hours: {len(COMMON):,} of {len(TEST_HOURS):,}")

### 1.3 Pick check

Each pick needs a `SPLIT` row in both model exports and must pass spec 09's bin rule in every `PICK_CATEGORIES` category. On a failure the notebook stops; after spec 09 is re-run with new picks, edit `PICKS`.

In [ ]:
SMARD_ROW = ("smard", "none")
VALUE = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="value")
COUNT = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="count")
HOURLY_ROWS = set(zip(hourly["model"], hourly["split_method"]))


def failed_rules(row, category):
    """Spec 09's bin rule: the rules a row fails in a category; empty if it qualifies."""
    failed = []
    for view in ["actual", "forecast"]:
        metric = f"MAE_{category}_by_{view}"
        if not VALUE.loc[row, metric] < VALUE.loc[SMARD_ROW, metric]:
            failed.append(f"{metric} {VALUE.loc[row, metric]:,.0f} not below SMARD's {VALUE.loc[SMARD_ROW, metric]:,.0f}")
    by_actual = COUNT.loc[row, f"MAE_{category}_by_actual"]
    share = COUNT.loc[row, f"MAE_{category}_by_forecast"] / by_actual if by_actual else np.nan
    if not share >= MIN_BIN_SHARE:
        failed.append(f"forecast share {share:.2f} < {MIN_BIN_SHARE}")
    return failed


problems = []
for direction, model in PICKS.items():
    row = (model, SPLIT)
    if row not in HOURLY_ROWS:
        problems.append(f"{direction} {model}: no {SPLIT} row in model_forecast_errors_hourly.csv")
    if row not in VALUE.index:
        problems.append(f"{direction} {model}: no {SPLIT} row in model_scoreboard.csv")
        continue
    for category in PICK_CATEGORIES[direction]:
        failed = failed_rules(row, category)
        problems += [f"{direction} {model} in {category}: {rule}" for rule in failed]
        by_actual, by_forecast = (f"MAE_{category}_by_{view}" for view in ["actual", "forecast"])
        print(
            f"{direction:<4} {model:<20} {category:<12} "
            f"MAE by actual {VALUE.loc[row, by_actual]:,.0f} (SMARD {VALUE.loc[SMARD_ROW, by_actual]:,.0f}), "
            f"by forecast {VALUE.loc[row, by_forecast]:,.0f} (SMARD {VALUE.loc[SMARD_ROW, by_forecast]:,.0f}), "
            f"forecast share {COUNT.loc[row, by_forecast] / COUNT.loc[row, by_actual]:.2f}: "
            f"{'fails' if failed else 'passes'}"
        )

if problems:
    raise RuntimeError(
        f"pick check failed for PICKS = {PICKS}:\n  " + "\n  ".join(problems)
        + "\nRe-run visualization-01-regression-best-models.ipynb and edit PICKS; no model is picked here."
    )
print("pick check passed")

### 1.4 Snapshot self-check

Labels, model exports and SMARD file must come from one `smard.csv` fetch; the notebook never joins on a partial overlap.

In [ ]:
REGENERATE = "API-connection → risk-definition → forecast-metrics-claude → regression-models-claude"


def require(ok, problem):
    """Stop the notebook, naming the regeneration order."""
    if not ok:
        raise RuntimeError(f"self-check failed: {problem}. Regenerate in this order: {REGENERATE}.")


require(len(COMMON) > 0, "no common hours")

# Every test day and hour has a label row
missing_days = TEST_DAYS.difference(labels_daily.index)
require(missing_days.empty, f"{len(missing_days)} test days missing from risk_labels_daily.csv")
missing_hours = TEST_HOURS.difference(labels_hourly.index)
require(missing_hours.empty, f"{len(missing_hours)} test hours missing from risk_labels_hourly.csv")

# Same actuals in the three hourly files
for name, other in [
    ("smard_forecast_errors_hourly.csv", smard["residual_load"]),
    ("risk_labels_hourly.csv", labels_hourly["residual_load"]),
]:
    gap = float((ACTUAL[COMMON] - other.reindex(COMMON)).abs().max())
    require(gap <= 1e-6, f"residual_load of the model export and {name} differ by up to {gap:,.2f} MWh")

print(
    f"self-check passed: labels, model exports and SMARD file share one snapshot "
    f"({len(TEST_DAYS)} days, {len(COMMON):,} common hours)"
)

## 2 Labels

### 2.1 Rule functions

Copied from `risk-definition.ipynb` §3.1, §3.2 and §5.1. One change: `day_rules` counts a day only when it is evaluable (2.2), in place of `day_complete`.

In [ ]:
# Resolution is measured, not assumed.
RESOLUTION = TEST_HOURS.to_series().diff().mode().iloc[0]
MIN_RUN = int(PERSISTENCE / RESOLUTION)
EXPECTED_OBS_PER_DAY = int(pd.Timedelta("1D") / RESOLUTION)
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))

# The rules run on the common hours of the test days
DAY = pd.Series(COMMON.normalize(), index=COMMON)
DAYS = TEST_DAYS


def hourly_crossings(series, threshold_by_day, direction):
    """Per-observation crossing flags against each observation's own day's threshold.

    Returns nullable booleans: pd.NA wherever the day's threshold is undefined, so an undefined
    threshold can never masquerade as "not at risk".
    """
    thr = pd.Series(
        threshold_by_day.reindex(series.index.normalize()).to_numpy(), index=series.index
    )
    crossed = (series >= thr) if direction == "high" else (series <= thr)
    return crossed.astype("boolean").mask(thr.isna())


def day_rules(hourly_flag, threshold_by_day):
    """The `any` and `3h` day rules from per-observation crossing flags.

    `any`  — at least one observation crosses.
    `3h`   — the crossing persists for at least PERSISTENCE, consecutively, within the day.

    Consecutive *rows* count as consecutive: on the spring-DST day the missing 02:00 makes
    a run spanning it one hour longer in wall-clock terms than in rows. The approximation is noted
    rather than corrected.

    Days with an undefined threshold, or not evaluable, are pd.NA in both rules rather than False.
    """
    filled = hourly_flag.fillna(False).astype(bool)
    any_rule = filled.groupby(DAY).any().reindex(DAYS, fill_value=False)

    run_id = (filled != filled.shift()).cumsum()
    run_len = filled.groupby([DAY, run_id]).transform("size").where(filled, 0)
    persist_rule = (run_len.groupby(DAY).max() >= MIN_RUN).reindex(DAYS, fill_value=False)

    valid = (threshold_by_day.notna() & EVALUABLE).reindex(DAYS, fill_value=False)
    return any_rule.astype("boolean").mask(~valid), persist_rule.astype("boolean").mask(~valid)


def flagged_range(hourly_flag, rule):
    """Earliest and latest flagged observation per day, as a span (see §5.1).

    Under the `3h` rule only observations belonging to a run of at least PERSISTENCE count.
    """
    filled = hourly_flag.fillna(False).astype(bool)
    if rule == "3h":
        run_id = (filled != filled.shift()).cumsum()
        run_len = filled.groupby([DAY, run_id]).transform("size")
        filled = filled & (run_len >= MIN_RUN)

    stamps = pd.Series(filled.index[filled], index=filled.index[filled].normalize())
    grouped = stamps.groupby(level=0)
    return grouped.min().reindex(DAYS), grouped.max().reindex(DAYS)

### 2.2 Flags per source

The actual's flags and ranges are read from the label files; the model's and SMARD's come from the rule functions against the exported thresholds. A day is evaluable when its actual labels are set and its common hours cover `DAY_COMPLETENESS`; on any other day every flag is empty for every source. The two frames built here are the export layout of §6 and the only input of §3–§5.

In [ ]:
FLAG_SOURCES = ["actual", "model", "smard"]


def is_set(flag):
    """True where a flag is set; False and empty both give False. A new mask, the label stays three-state."""
    return pd.Series(flag.to_numpy(dtype=bool, na_value=False), index=flag.index)


THRESHOLD = {(d, b): labels_daily[f"{d}_threshold_{b}"].reindex(TEST_DAYS) for d, b in LABEL_PAIRS}

# Evaluable: actual labels set and enough common hours
common_per_day = pd.Series(1, index=COMMON).groupby(COMMON.normalize()).size().reindex(TEST_DAYS, fill_value=0)
EVALUABLE = labels_daily.loc[TEST_DAYS, DAILY_FLAGS].notna().all(axis=1) & common_per_day.ge(MIN_OBS_PER_DAY)
HOUR_EVALUABLE = EVALUABLE.reindex(COMMON.normalize()).to_numpy()

# The hourly series each source is flagged on
SERIES = {
    "actual": {"high": ACTUAL[COMMON], "low": ACTUAL[COMMON]},
    "model": {d: FORECAST.loc[COMMON, PICKS[d]] for d in BASES},
    "smard": {"high": SMARD_FORECAST[COMMON], "low": SMARD_FORECAST[COMMON]},
}


def rule_flags(series, direction, basis):
    """Hour flags, day flags and flagged ranges of a series against the exported threshold."""
    threshold = THRESHOLD[(direction, basis)]
    hour = hourly_crossings(series, threshold, direction).mask(~HOUR_EVALUABLE)
    any_rule, persist_rule = day_rules(hour, threshold)
    return hour, {"any": any_rule, "3h": persist_rule}, {rule: flagged_range(hour, rule) for rule in RULES}


def file_flags(direction, basis):
    """The same three, read from the label files."""
    hour = labels_hourly.loc[COMMON, f"{direction}_risk_hour_{basis}"].mask(~HOUR_EVALUABLE)
    days = {rule: labels_daily.loc[TEST_DAYS, f"{direction}_risk_{basis}_{rule}"].mask(~EVALUABLE) for rule in RULES}
    ranges = {
        rule: tuple(labels_daily.loc[TEST_DAYS, f"{direction}_range_{edge}_{basis}_{rule}"].where(EVALUABLE) for edge in ("start", "end"))
        for rule in RULES
    }
    return hour, days, ranges


day_columns = {
    "evaluable": EVALUABLE,
    "high_model": PICKS["high"],
    "low_model": PICKS["low"],
    **{f"{d}_threshold_{b}": THRESHOLD[(d, b)] for d, b in LABEL_PAIRS},
}
by_day = lambda series: series.groupby(series.index.normalize())
for source in FLAG_SOURCES:
    day_columns[f"{source}_max"] = by_day(SERIES[source]["high"]).max()
    day_columns[f"{source}_min"] = by_day(SERIES[source]["low"]).min()
hour_columns = {
    "residual_load": ACTUAL[COMMON],
    "high_model_forecast": SERIES["model"]["high"],
    "low_model_forecast": SERIES["model"]["low"],
    "fc_residual_load": SMARD_FORECAST[COMMON],
}
flag_columns, range_columns = {}, {}
for source in FLAG_SOURCES:
    for d, b in LABEL_PAIRS:
        hour, days, ranges = file_flags(d, b) if source == "actual" else rule_flags(SERIES[source][d], d, b)
        hour_columns[f"{source}_{d}_risk_hour_{b}"] = hour
        for rule in RULES:
            flag_columns[f"{source}_{d}_risk_{b}_{rule}"] = days[rule]
            for edge, values in zip(("start", "end"), ranges[rule]):
                range_columns[f"{source}_{d}_range_{edge}_{b}_{rule}"] = values

DAILY_EXPORT = pd.DataFrame(day_columns | flag_columns | range_columns, index=TEST_DAYS).rename_axis("date")
HOURLY_EXPORT = pd.DataFrame(hour_columns, index=COMMON).rename_axis("timestamp")

print(f"evaluable days: {EVALUABLE.sum()} of {len(TEST_DAYS)} test days ({(~EVALUABLE).sum()} excluded)")
counts = pd.DataFrame(
    {
        (source, column): [
            int(DAILY_EXPORT[f"{source}_{d}_risk_{b}_{rule}"].sum()) if column != "hours"
            else int(HOURLY_EXPORT[f"{source}_{d}_risk_hour_{b}"].sum())
            for d, b in LABEL_PAIRS
        ]
        for source in FLAG_SOURCES
        for rule, column in [("any", "any days"), ("3h", "3h days"), ("any", "hours")]
    },
    index=pd.MultiIndex.from_tuples(LABEL_PAIRS, names=["direction", "basis"]),
)
counts

### 2.3 Rule reproduction check

On every test day whose hours are all common, the rule functions applied to the actual must reproduce the label files exactly: day flags, ranges and hour flags. On a day that lost a common hour the file's flag may rest on an hour the forecasts don't have, so such days are counted, not checked.

In [ ]:
hours_per_day = pd.Series(1, index=TEST_HOURS).groupby(TEST_HOURS.normalize()).size()
FULLY_COMMON = TEST_DAYS[common_per_day.eq(hours_per_day).to_numpy()]
in_full = COMMON.normalize().isin(FULLY_COMMON)


def differs(a, b):
    """Where two series disagree; two empty cells agree."""
    one_empty = a.isna() != b.isna()
    return one_empty | (a.notna() & b.notna() & (a != b)).astype(bool)


problems = []
for d, b in LABEL_PAIRS:
    hour, days, ranges = rule_flags(ACTUAL[COMMON], d, b)
    pairs = {f"{d}_risk_hour_{b}": (hour[in_full], labels_hourly.loc[COMMON[in_full], f"{d}_risk_hour_{b}"])}
    for rule in RULES:
        pairs[f"{d}_risk_{b}_{rule}"] = (days[rule], labels_daily[f"{d}_risk_{b}_{rule}"])
        for edge, values in zip(("start", "end"), ranges[rule]):
            pairs[f"{d}_range_{edge}_{b}_{rule}"] = (values, labels_daily[f"{d}_range_{edge}_{b}_{rule}"])
    for column, (rebuilt, exported) in pairs.items():
        index = rebuilt.index if column.endswith(f"hour_{b}") else FULLY_COMMON
        wrong = int(differs(rebuilt.reindex(index), exported.reindex(index)).sum())
        if wrong:
            problems.append(f"{column}: {wrong} rows differ")

if problems:
    raise RuntimeError(
        "rule reproduction check failed; the copied rule functions no longer match risk-definition.ipynb:\n  "
        + "\n  ".join(problems)
    )
print(
    f"rule reproduction check passed on {len(FULLY_COMMON)} fully common test days "
    f"({len(TEST_DAYS) - len(FULLY_COMMON)} days lost a common hour and are not checked)"
)

### 2.4 Structural checks

On every source: `3h` implies `any`, a range exists exactly where the day is flagged, the day `any` flag equals "at least one flagged hour", and a flag is empty exactly where the day is not evaluable. §6 repeats them on the written files. Days flagged in both directions are reported, not prevented (`risk-definition.ipynb` §4.2); a non-zero count hints at a join error.

In [ ]:
def structural_problems(daily, hourly):
    """The Behaviour 9 checks on export-layout frames; empty if all pass."""
    problems = []
    evaluable = daily["evaluable"].astype(bool)
    for source in FLAG_SOURCES:
        for d, b in LABEL_PAIRS:
            flag = {rule: daily[f"{source}_{d}_risk_{b}_{rule}"] for rule in RULES}
            hour = hourly[f"{source}_{d}_risk_hour_{b}"]
            hour_any = is_set(hour).groupby(hour.index.normalize()).any().reindex(daily.index, fill_value=False)
            hour_evaluable = evaluable.reindex(hour.index.normalize()).to_numpy()
            bad = {
                "3h without any": is_set(flag["3h"]) & ~is_set(flag["any"]),
                "any differs from its hours": evaluable & (is_set(flag["any"]) != hour_any),
                "hour flag set on a day not evaluable": pd.Series(hour.notna().to_numpy() & ~hour_evaluable),
            }
            for rule in RULES:
                start = daily[f"{source}_{d}_range_start_{b}_{rule}"]
                end = daily[f"{source}_{d}_range_end_{b}_{rule}"]
                bad[f"{rule} empty on an evaluable day or set on another"] = flag[rule].isna() == evaluable
                bad[f"{rule} range without flag or flag without range"] = is_set(flag[rule]) != (start.notna() & end.notna())
                bad[f"{rule} range with one edge"] = start.notna() != end.notna()
            problems += [f"{source} {d} {b}: {name} ({int(mask.sum())} rows)" for name, mask in bad.items() if mask.sum()]
    return problems


problems = structural_problems(DAILY_EXPORT, HOURLY_EXPORT)
if problems:
    raise RuntimeError("structural check failed:\n  " + "\n  ".join(problems))
print(f"structural checks passed on {len(FLAG_SOURCES)} sources × {len(LABEL_PAIRS)} direction-basis pairs")

# Days flagged in both directions, any basis and rule
for source in FLAG_SOURCES:
    flagged = {
        d: DAILY_EXPORT[[f"{source}_{d}_risk_{b}_{rule}" for b in BASES[d] for rule in RULES]].eq(True).any(axis=1)
        for d in BASES
    }
    print(f"days flagged in both directions, {source:<6}: {int((flagged['high'] & flagged['low']).sum())}")

## 3 Scores

Each forecast's flags against the actual's, per evaluable day and per common hour: a **hit** (both flagged), a **miss** (only the actual), a **false alarm** (only the forecast), or **quiet** (neither, the remainder).
`recall = hits / (hits + misses)`, `precision = hits / (hits + false alarms)`; an empty cell is a zero denominator. Hours have no `3h` rule, so each `3h` row repeats the hour counts of its `any` row.

In [ ]:
OUTCOMES = ["hit", "miss", "false alarm", "quiet"]
GROUPS = {"model": "model", "smard": "SMARD"}


def outcome(forecast, actual):
    """hit, miss, false alarm or quiet per row; empty where either flag is empty."""
    f, a = is_set(forecast), is_set(actual)
    labels = np.select([f & a, a & ~f, f & ~a], OUTCOMES[:3], OUTCOMES[3])
    return pd.Series(labels, index=actual.index).where(forecast.notna() & actual.notna())


def percent(part, whole):
    """part / whole in %, empty for a zero denominator."""
    return 100 * part / whole if whole else np.nan


rows = {}
for d, b in LABEL_PAIRS:
    for rule in RULES:
        row = {}
        for source, group in GROUPS.items():
            days = outcome(DAILY_EXPORT[f"{source}_{d}_risk_{b}_{rule}"], DAILY_EXPORT[f"actual_{d}_risk_{b}_{rule}"])
            hours = outcome(HOURLY_EXPORT[f"{source}_{d}_risk_hour_{b}"], HOURLY_EXPORT[f"actual_{d}_risk_hour_{b}"])
            hit, miss, false_alarm = (int(days.eq(name).sum()) for name in OUTCOMES[:3])
            row |= {
                (group, "actual days"): hit + miss,
                (group, "hit days"): hit,
                (group, "missed days"): miss,
                (group, "false-alarm days"): false_alarm,
                (group, "recall %"): percent(hit, hit + miss),
                (group, "precision %"): percent(hit, hit + false_alarm),
                **{(group, f"{name} hours"): int(hours.eq(outcome_name).sum())
                   for name, outcome_name in [("hit", "hit"), ("missed", "miss"), ("false-alarm", "false alarm")]},
            }
        rows[f"{d} {b} {rule}"] = row
SCORES = pd.DataFrame.from_dict(rows, orient="index")
SCORES.columns = pd.MultiIndex.from_tuples(SCORES.columns)

evaluable = DAILY_EXPORT["evaluable"]
print(
    f"test window {DAILY_EXPORT.index[0]:%Y-%m-%d} .. {DAILY_EXPORT.index[-1]:%Y-%m-%d}: "
    f"{evaluable.sum()} evaluable days, {(~evaluable).sum()} excluded (not evaluable); "
    f"model: high {DAILY_EXPORT['high_model'].iloc[0]}, low {DAILY_EXPORT['low_model'].iloc[0]} ({SPLIT} split)"
)
SCORES.style.format(
    {column: "{:.0f}" if column[1].endswith("%") else "{:,.0f}" for column in SCORES.columns}, na_rep=""
)

## 4 High extreme

### 4.1 Calendar

Each cell is a test day, showing the forecast's outcome under `any` on the `rolling` basis (model above, SMARD below): fill means the forecast flagged the day, outline means the actual did; a dot marks a day the actual sustained for at least 3 h (on a miss: a sustained event the forecast missed), a dashed border a public holiday.

In [ ]:
DIRECTION_LABEL = {"high": "High", "low": "Low"}
THRESHOLD_LABEL = {("high", "rolling"): "rolling P99", ("low", "rolling"): "rolling P1", ("low", "zero"): "zero (0 MWh)"}
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
HOLIDAYS = holidays.country_holidays(
    "DE", years=range(DAILY_EXPORT.index[0].year, DAILY_EXPORT.index[-1].year + 1), language="en_US"
)


def source_style(source, direction):
    """STYLE entry of a source: the direction's model, SMARD or the actual."""
    return STYLE[DAILY_EXPORT[f"{direction}_model"].iloc[0] if source == "model" else source]


def risk_calendar(direction):
    """Plot A: the day outcomes of the test year on an ISO-week grid, model above SMARD."""
    basis = "rolling"
    days = DAILY_EXPORT.index
    first_monday = days[0] - pd.Timedelta(days=days[0].weekday())
    n_weeks = (days[-1] - first_monday).days // 7 + 1
    column, row = np.asarray((days - first_monday).days // 7), np.asarray(days.weekday)
    actual = DAILY_EXPORT[f"actual_{direction}_risk_{basis}_any"]
    sustained = is_set(DAILY_EXPORT[f"actual_{direction}_risk_{basis}_3h"]).to_numpy()
    holiday = np.array([day in HOLIDAYS for day in days])
    tail = TAIL_COLOR[(direction, basis)]

    # Fill = forecast flagged, outline = actual flagged
    cell_style = {
        "hit": {"facecolor": tail, "edgecolor": tail, "linewidth": 2},
        "miss": {"facecolor": "white", "edgecolor": tail, "linewidth": 2},
        "false alarm": {"facecolor": OUTCOME_COLOR["quiet"], "edgecolor": tail, "linewidth": 0, "hatch": "////"},
        "quiet": {"facecolor": OUTCOME_COLOR["quiet"], "edgecolor": "none"},
        "not evaluable": {"facecolor": OUTCOME_COLOR["not evaluable"], "edgecolor": "#6B7480", "linewidth": 0.8, "hatch": "xxx"},
    }
    holiday_style = {"fill": False, "edgecolor": HOLIDAY_COLOR, "linewidth": 1.6, "linestyle": "--", "alpha": 0.8}
    sustained_style = {"marker": "o", "color": "#1C1C1C", "markersize": 3.5, "linestyle": "none"}

    with plt.rc_context({"hatch.linewidth": 1.4}):
        fig, axes = plt.subplots(2, 1, figsize=(15, 5.4), layout="constrained")
        for ax, source in zip(axes, ["model", "smard"]):
            result = outcome(DAILY_EXPORT[f"{source}_{direction}_risk_{basis}_any"], actual).fillna("not evaluable")
            for c, r, name, is_holiday in zip(column, row, result, holiday):
                inset = 0.14 if name in ("hit", "miss") else 0.1  # room for the outline
                ax.add_patch(Rectangle((c + inset, r + inset), 1 - 2 * inset, 1 - 2 * inset, **cell_style[name]))
                if is_holiday:
                    ax.add_patch(Rectangle((c + 0.02, r + 0.02), 0.96, 0.96, **holiday_style))
            ax.plot(column[sustained] + 0.5, row[sustained] + 0.5, zorder=3, **sustained_style)

            ax.set_xlim(0, n_weeks)
            ax.set_ylim(7, 0)
            ax.set_aspect("equal")
            ax.set_xticks([])
            ax.set_yticks(np.arange(7) + 0.5, WEEKDAYS)
            ax.tick_params(length=0)
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.set_title(source_style(source, direction)["label"], loc="left", fontsize=12, pad=16)

            # Per panel: the three outcomes that differ between forecasts, with day counts
            counts = result.value_counts()
            handles = [Patch(**cell_style[name], label=f"{name} ({counts.get(name, 0)})") for name in ["hit", "miss", "false alarm"]]
            ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=10)

        # Month labels above the top panel, at the week holding the 1st
        for month_start in pd.date_range(days[0], days[-1], freq="MS"):
            label = f"{month_start:%b %Y}" if month_start.month == 1 else f"{month_start:%b}"
            axes[0].text((month_start - first_monday).days // 7 + 0.5, -0.25, label, ha="left", va="bottom", fontsize=9, color="grey")
        # One line under the bottom panel: the first and last test day at their columns, the axis name between
        below = {"transform": axes[1].get_xaxis_transform(), "va": "top", "fontsize": 10, "color": "grey"}
        axes[1].text(column[0] + 0.1, -0.06, f"{days[0]:%d %b %Y}", ha="left", **below)
        axes[1].text(n_weeks / 2, -0.06, "ISO week (Monday start)", ha="center", **below)
        axes[1].text(column[-1] + 0.9, -0.06, f"{days[-1]:%d %b %Y}", ha="right", **below)

        # Shared by both panels, under the plot; "not evaluable" only if such days exist
        not_evaluable = int((~DAILY_EXPORT["evaluable"]).sum())
        shared = [Patch(**cell_style["quiet"], label="quiet (neither flagged)")]
        if not_evaluable:
            shared.append(Patch(**cell_style["not evaluable"], label=f"not evaluable ({not_evaluable})"))
        shared += [
            Line2D([], [], **sustained_style, label="actual sustained ≥ 3 h (3h rule)"),
            Patch(**holiday_style, label="public holiday"),
        ]
        fig.legend(handles=shared, loc="lower center", bbox_to_anchor=(0.45, -0.07), ncol=4, frameon=False, fontsize=10)
        fig.suptitle(
            f"{DIRECTION_LABEL[direction]} risk days of the test year: outcome under any, {THRESHOLD_LABEL[(direction, basis)]}",
            fontsize=15,
        )
        plt.show()


risk_calendar("high")

### 4.2 Zoom weeks

Two test weeks, hourly: the three series against the day's threshold (dash-dot), and below them the hours each source flags (full bar: part of a run of at least 3 h; low bar: `any` only). Dotted guides mark 06:00, 12:00 and 18:00.

In [ ]:
def week_of(days):
    """The Monday of each day's ISO week (a day or an index)."""
    return days.to_period("W-SUN").start_time


def lowest_flagged_day(months, holidays_only):
    """Monday of the week of the flagged low rolling day with the lowest actual day minimum, or None."""
    days = DAILY_EXPORT[is_set(DAILY_EXPORT["actual_low_risk_rolling_any"]) & DAILY_EXPORT.index.month.isin(months)]
    if holidays_only:
        days = days[[day in HOLIDAYS for day in days.index]]
    return None if days.empty else week_of(days["actual_min"].idxmin())


def most_actual_high_days(_):
    """Monday of the week with the most actual high rolling any days (ties: highest day maximum), or None."""
    flagged = is_set(DAILY_EXPORT["actual_high_risk_rolling_any"])
    weeks = pd.DataFrame({"days": flagged, "peak": DAILY_EXPORT["actual_max"]}).groupby(week_of(DAILY_EXPORT.index))
    table = weeks.agg(days=("days", "sum"), peak=("peak", "max")).sort_values(["days", "peak"], ascending=False)
    return table.index[0] if table["days"].iloc[0] > 0 else None


def most_high_errors(excluded):
    """Monday of the week with the most misses and false alarms of model and SMARD (ties: earliest), or None."""
    actual = DAILY_EXPORT["actual_high_risk_rolling_any"]
    errors = sum(
        outcome(DAILY_EXPORT[f"{source}_high_risk_rolling_any"], actual).isin(["miss", "false alarm"]).astype(int)
        for source in GROUPS
    )
    per_week = errors.groupby(week_of(DAILY_EXPORT.index)).sum().drop(excluded, errors="ignore")
    return per_week.idxmax() if per_week.max() > 0 else None  # idxmax: the earliest on ties


ZOOM_RULES = {
    "high": [
        ("the most actual high rolling any days (ties: highest day maximum)", most_actual_high_days),
        ("the most misses and false alarms of model and SMARD, panel 1 excluded (ties: earliest)", most_high_errors),
    ],
    "low": [
        ("the flagged low rolling public holiday in Mar–Aug with the lowest day minimum",
         lambda _: lowest_flagged_day([3, 4, 5, 6, 7, 8], holidays_only=True)),
        ("the flagged low rolling day in Dec–Feb with the lowest day minimum",
         lambda _: lowest_flagged_day([12, 1, 2], holidays_only=False)),
    ],
}


def zoom_weeks(direction):
    """(Monday or None, reason) per panel: ZOOM_WEEKS if set, else the zoom rule."""
    weeks = []
    for setting, (rule, pick) in zip(ZOOM_WEEKS[direction], ZOOM_RULES[direction]):
        if setting is not None:
            monday = pd.Timestamp(setting)
            if monday.weekday() != 0:
                raise ValueError(f"ZOOM_WEEKS[{direction!r}]: {setting} is not a Monday")
            weeks.append((monday, "set in ZOOM_WEEKS"))
        else:
            monday = pick(weeks[0][0] if weeks else None)
            weeks.append((monday, rule) if monday is not None else (None, f"no week meets: {rule}"))
    return weeks


def persistent_hours(flag):
    """Flagged hours that belong to a run of at least PERSISTENCE within their day."""
    filled = is_set(flag)
    run_id = (filled != filled.shift()).cumsum()
    run_len = filled.groupby([flag.index.normalize(), run_id]).transform("size")
    return filled & run_len.ge(MIN_RUN)


def flagged_span(source, direction, basis, day):
    """The day's any-range of a source as text, e.g. 16:00–20:00, or — if not flagged."""
    start = DAILY_EXPORT.loc[day, f"{source}_{direction}_range_start_{basis}_any"]
    end = DAILY_EXPORT.loc[day, f"{source}_{direction}_range_end_{basis}_any"]
    if pd.isna(start):
        return "—"
    return f"{start:%H:%M}" if start == end else f"{start:%H:%M}–{end:%H:%M}"


def event_zoom(direction):
    """Plot B: two test weeks, hourly, with the flag strip of each source."""
    basis = "rolling"
    if not is_set(DAILY_EXPORT[f"actual_{direction}_risk_{basis}_any"]).any():
        print(f"{direction}: the actual flags no {basis} day in the test window, so there is no week to zoom into")
        return
    tail = TAIL_COLOR[(direction, basis)]
    sources = ["actual", "model", "smard"]

    threshold_style = {"color": tail, "linewidth": 1.8, "linestyle": (0, (6, 2, 1, 2))}  # dash-dot, apart from the model line
    zero_style = {"color": TAIL_COLOR[("low", "zero")], "linestyle": ":", "linewidth": 2}
    long_bar = {"height": 0.8, "color": tail}                  # hour in a run of at least PERSISTENCE
    short_bar = {"height": 0.26, "color": tail, "alpha": 0.45}  # hour flagged under any only

    fig = plt.figure(figsize=(15, 11), layout="constrained")
    series_handles = []
    grid = fig.add_gridspec(4, 1, height_ratios=[4, 1, 4, 1])
    for panel, (monday, reason) in enumerate(zoom_weeks(direction)):
        ax = fig.add_subplot(grid[2 * panel])
        strip = fig.add_subplot(grid[2 * panel + 1], sharex=ax)
        print(f"panel {panel + 1}: " + (f"week of {monday:%a %d %b %Y}, {reason}" if monday is not None else reason))
        if monday is None:
            ax.text(0.5, 0.5, reason, ha="center", va="center", transform=ax.transAxes, color="grey")
            ax.set_axis_off()
            strip.set_axis_off()
            continue

        sunday_end = monday + pd.Timedelta(days=7)
        week = HOURLY_EXPORT[(HOURLY_EXPORT.index >= monday) & (HOURLY_EXPORT.index < sunday_end)]
        series = {"actual": week["residual_load"], "model": week[f"{direction}_model_forecast"], "smard": week["fc_residual_load"]}
        for source in sources:
            style = source_style(source, direction)
            ax.plot(week.index, series[source], color=style["color"], linestyle=style.get("linestyle", "-"),
                    linewidth=1.4, label=style["label"])
        if not series_handles:
            series_handles = ax.get_legend_handles_labels()[0]
        threshold = DAILY_EXPORT[f"{direction}_threshold_{basis}"].reindex(week.index.normalize()).to_numpy()
        ax.plot(week.index, threshold, drawstyle="steps-post", zorder=1, **threshold_style)
        if direction == "low":
            ax.axhline(0, zorder=1, **zero_style)

        # Direct labels at the same spot in both panels: high above the line at Monday 01:00,
        # low outside the right edge (two lines close together near 0 MWh)
        threshold_text = f"threshold {THRESHOLD_LABEL[(direction, basis)].split()[-1]}"
        if direction == "high":
            ax.annotate(threshold_text, (monday + pd.Timedelta(hours=1), threshold[0]), xytext=(0, 4), textcoords="offset points",
                        ha="left", va="bottom", fontsize=10, color="#1C1C1C")
        else:
            for value, text in [(threshold[-1], threshold_text), (0, "0 MWh (zero basis)")]:
                ax.annotate(text, (sunday_end, value), xytext=(6, 0), textcoords="offset points", ha="left", va="center",
                            fontsize=10, color="#1C1C1C", annotation_clip=False)

        # Flag strip: a full solid bar in a 3h run, a low faint bar for any only
        for row, source in enumerate(sources):
            flag = week[f"{source}_{direction}_risk_hour_{basis}"]
            long = persistent_hours(flag)
            short = is_set(flag) & ~long
            for hours, bar in [(long, long_bar), (short, short_bar)]:
                strip.bar(week.index[hours.to_numpy()], bottom=row - bar["height"] / 2, width=1 / 24, align="edge", linewidth=0, zorder=2, **bar)

        # One tick per day at noon, a holiday's name as a second line
        days = pd.date_range(monday, periods=7, freq="D")
        labels = [f"{day:%a %d %b}" + (f"\n{HOLIDAYS.get(day)}" if day in HOLIDAYS else "") for day in days]
        style_timeseries(ax, f"Week of {monday:%d %b %Y}", "Residual Load [MWh]")
        ax.set_xticks(days + pd.Timedelta(hours=12), labels)
        ax.set_xlim(monday, sunday_end)
        ax.tick_params(labelbottom=False)
        # Main axes: grey day separators and dotted guides at 06:00, 12:00 and 18:00
        # Strip: the same in white on the grey band, under the bars, plus white lines between the rows
        for day in days:
            if day > monday:
                ax.axvline(day, color="0.7", linewidth=0.9, zorder=0)
                strip.axvline(day, color="white", linewidth=2.2, zorder=1)
            for hour in (6, 12, 18):
                ax.axvline(day + pd.Timedelta(hours=hour), color="0.72", linewidth=0.8, linestyle=(0, (1, 3)), zorder=0)
                strip.axvline(day + pd.Timedelta(hours=hour), color="white", linewidth=0.9, zorder=1)
        for boundary in np.arange(len(sources) - 1) + 0.5:
            strip.axhline(boundary, color="white", linewidth=2.2, zorder=1)
        strip.set_ylim(len(sources) - 0.5, -0.5)
        strip.set_yticks(range(len(sources)), ["actual", "model", "SMARD"])
        strip.tick_params(length=0)
        strip.set_facecolor(OUTCOME_COLOR["quiet"])
        for spine in strip.spines.values():
            spine.set_visible(False)

        # Definition 2 sentence per flagged day
        flag_columns = [f"{s}_{direction}_risk_{basis}_any" for s in sources]
        flagged = [day for day in days if day in DAILY_EXPORT.index and DAILY_EXPORT.loc[day, flag_columns].eq(True).any()]
        for day in flagged:
            print(f"  {day:%a %d %b}: " + ", ".join(
                f"{name} {flagged_span(source, direction, basis, day)}" for source, name in zip(sources, ["actual", "model", "SMARD"])
            ))
        if not flagged:
            print("  no flagged day")

    # Two legend groups: the series left, the risk label right
    label_handles = [
        Patch(color=tail, label="flagged hour, part of a run ≥ 3 h"),
        Patch(color=tail, alpha=short_bar["alpha"], label="flagged hour, any only (low bar)"),
    ]
    common = {"frameon": False, "fontsize": 10, "title_fontsize": 10, "alignment": "left"}
    fig.legend(handles=series_handles, title="Residual load", loc="upper right", bbox_to_anchor=(0.48, 0), ncol=3, **common)
    fig.legend(handles=label_handles, title="Risk label", loc="upper left", bbox_to_anchor=(0.52, 0), ncol=2, **common)
    fig.suptitle(f"Risk: {DIRECTION_LABEL[direction]} Extreme (two test weeks)", fontsize=15)
    plt.show()


event_zoom("high")

### 4.3 Day margin

One dot per evaluable day: how far the day's maximum lies past the threshold, actual (x) against forecast (y). Top right (tinted) holds the hits, bottom right (outlined) the misses, top left (hatched) the false alarms, bottom left the ordinary days; filled dots are days the actual sustained for 3 h, hollow ones `any` only, small ringed ones days the actual did not flag (faint on ordinary days).

In [ ]:
def day_margin(direction):
    """Plot C: day extreme minus threshold, forecast against actual; SMARD, then model, one row per basis."""
    daily = DAILY_EXPORT[DAILY_EXPORT["evaluable"]]
    bases = BASES[direction]
    if not any(is_set(daily[f"actual_{direction}_risk_{b}_any"]).any() for b in bases):
        print(f"{direction}: the actual flags no day in the test window, so there is no margin to compare")
        return
    extreme = "max" if direction == "high" else "min"
    sources = ["smard", "model"]

    # Margins per basis and source; flagged means margin >= 0 (high) or <= 0 (low)
    sign = 1 if direction == "high" else -1
    margin = {
        (b, source): (
            daily[f"actual_{extreme}"] - daily[f"{direction}_threshold_{b}"],
            daily[f"{source}_{extreme}"] - daily[f"{direction}_threshold_{b}"],
        )
        for b in bases for source in sources
    }
    inside = {key: (x.abs() <= MARGIN_WINDOW) & (y.abs() <= MARGIN_WINDOW) for key, (x, y) in margin.items()}
    lim = 1.08 * max(max(x[inside[key]].abs().max(), y[inside[key]].abs().max()) for key, (x, y) in margin.items())

    # Fixed label spots as (actual, forecast) fractions of the axis, + = flagged side
    LABEL_SPOT = {"hits": (0.5, 0.8), "misses": (0.5, -0.8), "false alarms": (-0.5, 0.8)}  # x: middle of the quadrant
    quadrant_styles = {
        b_tail: {
            "hits": {"facecolor": b_tail, "alpha": 0.25, "linewidth": 0},
            "misses": {"fill": False, "edgecolor": b_tail, "linewidth": 2.5},
            "false alarms": {"fill": False, "edgecolor": b_tail, "linewidth": 0, "hatch": "////", "alpha": 0.22},
        }
        for b_tail in {TAIL_COLOR[(direction, b)] for b in bases}
    }

    fig, axes = plt.subplots(len(bases), 2, figsize=(12, 6 * len(bases)), squeeze=False, layout="constrained")
    for (b, source), ax in zip(margin, axes.flat):
        axes_row = axes[bases.index(b)]
        x, y = margin[(b, source)]
        style = source_style(source, direction)
        tail = TAIL_COLOR[(direction, b)]
        quadrant_style = quadrant_styles[tail]
        hit_x, hit_y = sign * x >= 0, sign * y >= 0
        counts = {"hits": hit_x & hit_y, "misses": hit_x & ~hit_y, "false alarms": ~hit_x & hit_y}

        # The scores table must count the same days
        expected = SCORES.loc[f"{direction} {b} any", GROUPS[source]]
        for name, column in [("hits", "hit days"), ("misses", "missed days"), ("false alarms", "false-alarm days")]:
            if counts[name].sum() != expected[column]:
                raise RuntimeError(f"day margin {direction} {b} {source}: {counts[name].sum()} {name}, scores table {expected[column]:.0f}")

        # Quadrants in the calendar's styles: hits tinted, misses outlined, false alarms hatched
        ax.add_patch(Rectangle((0, 0), sign * lim, sign * lim, **quadrant_style["hits"]))
        ax.add_patch(Rectangle((0, 0), sign * 0.99 * lim, -sign * 0.99 * lim, **quadrant_style["misses"]))  # inset: outline not clipped
        ax.add_patch(Rectangle((0, 0), -sign * lim, sign * lim, **quadrant_style["false alarms"]))
        ax.axhline(0, color="0.6", linewidth=0.8)
        ax.axvline(0, color="0.6", linewidth=0.8)
        ax.plot([-lim, lim], [-lim, lim], color="0.75", linewidth=0.8, linestyle="--")

        shown = inside[(b, source)]
        sustained = is_set(daily[f"actual_{direction}_risk_{b}_3h"])
        any_only = is_set(daily[f"actual_{direction}_risk_{b}_any"]) & ~sustained
        quiet = ~(sustained | any_only)
        false_alarm = quiet & counts["false alarms"]
        ax.scatter(x[shown & quiet & ~false_alarm], y[shown & quiet & ~false_alarm], s=8, color=style["color"], alpha=0.35, linewidth=0)
        ax.scatter(x[shown & false_alarm], y[shown & false_alarm], s=22, color=style["color"], edgecolor="white", linewidth=0.8, zorder=3)
        ax.scatter(x[shown & any_only], y[shown & any_only], s=30, facecolor="none", edgecolor=style["color"], linewidth=1.2)
        ax.scatter(x[shown & sustained], y[shown & sustained], s=30, color=style["color"], linewidth=0)

        # Count labels at the same spot in every panel, away from the diagonal: big number, small word
        glass = {"boxstyle": "round,pad=0,rounding_size=0.06", "facecolor": "#F5F7FA", "edgecolor": "#CBD2DA", "linewidth": 0.6, "alpha": 0.9}
        hits, misses = counts["hits"].sum(), counts["misses"].sum()
        for name, (u, v) in LABEL_SPOT.items():
            spot = (sign * u * lim, sign * v * lim)
            number = f"{counts[name].sum()}"
            # Tile centred 16 pt above the spot, sized in points; the number sits optically centred in it
            width, height, centre = max(26, 13 * len(number) + 12), 28, 16
            at_spot = fig.dpi_scale_trans + ScaledTranslation(*spot, ax.transData)
            ax.add_patch(FancyBboxPatch((-width / 144, (centre - height / 2) / 72), width / 72, height / 72,
                                        transform=at_spot, zorder=4, **glass))
            ax.annotate(number, spot, xytext=(0, centre - 0.36 * 20), textcoords="offset points", ha="center", va="baseline",
                        fontsize=20, fontweight="bold", color="#1C1C1C", zorder=5)
            ax.annotate(name, spot, xytext=(0, -2), textcoords="offset points", ha="center", va="top", fontsize=10, color="#48505A")
            if name == "hits" and hits + misses:
                ax.annotate(f"recall {100 * hits / (hits + misses):.0f} %", spot, xytext=(0, -15), textcoords="offset points",
                            ha="center", va="top", fontsize=8.5, color="#48505A")
        ax.annotate("ordinary", (-sign * 0.5 * lim, -sign * 0.93 * lim), ha="center", va="center", fontsize=9, color="grey", style="italic")

        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect("equal")
        style_plain(ax, f"{style['label']}, {THRESHOLD_LABEL[(direction, b)]}",
                    f"forecast day {extreme} − threshold (MWh)", f"actual day {extreme} − threshold (MWh)")
        ax.xaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
        if ax is not axes_row[0]:
            ax.set_ylabel("")  # the left panel names the y axis
        print(f"{b} {source}: {(~shown).sum()} of {len(shown)} evaluable days lie outside ±{MARGIN_WINDOW:,} MWh and are not drawn")

    handles = [
        Line2D([], [], marker="o", linestyle="none", color="#48505A", markersize=6, label="actual flagged 3h"),
        Line2D([], [], marker="o", linestyle="none", markerfacecolor="none", markeredgecolor="#48505A", markersize=6, label="actual flagged any only"),
        Line2D([], [], marker="o", linestyle="none", color="#48505A", markeredgecolor="white", markersize=5, label="actual not flagged (faint: ordinary day)"),
    ]
    tail = TAIL_COLOR[(direction, bases[0])]
    handles += [
        Patch(**quadrant_styles[tail]["hits"], label="hits (both flagged)"),
        Patch(**quadrant_styles[tail]["misses"], label="misses (actual only)"),
        Patch(**quadrant_styles[tail]["false alarms"], label="false alarms (forecast only)"),
    ]
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0), ncol=3, frameon=False, fontsize=10)
    fig.suptitle(f"{DIRECTION_LABEL[direction]} risk: day {extreme} against the threshold, forecast vs actual", fontsize=15)
    with plt.rc_context({"hatch.linewidth": 1.2}):
        plt.show()


day_margin("high")

## 5 Low extreme

### 5.1 Calendar

Read as in 4.1, on the low `rolling` basis (P1): fill means the forecast flagged the day, outline means the actual did; the `zero` basis appears in 5.2 and 5.3.

In [ ]:
risk_calendar("low")

### 5.2 Zoom weeks

Read as in 4.2: the dash-dot line is the `rolling` P1 threshold, the dotted line 0 MWh (`zero` basis); the strip shows the `rolling` flags.

In [ ]:
event_zoom("low")

### 5.3 Day margin

Read as in 4.3 with the day minimum, mirrored: the low tail lies bottom left, so the hits sit bottom left (tinted), the misses top left (outlined), the false alarms bottom right (hatched) and the ordinary days top right; one row per basis, `rolling` above `zero`.

In [ ]:
day_margin("low")

## 6 Export

### 6.1 Label files

Written only when `EXPORT_ENABLED` is on, in the layout of `risk_labels_*.csv` with a source prefix (`actual`, `model`, `smard`); an empty flag means not evaluable, never "not at risk".

In [ ]:
EXPORT_DIR = DATA_DIR / "risk_classification"
EXPORT_PATHS = {
    "daily": EXPORT_DIR / "model_risk_labels_daily.csv",
    "hourly": EXPORT_DIR / "model_risk_labels_hourly.csv",
}

if EXPORT_ENABLED:
    # Dates as %Y-%m-%d, timestamps and ranges as %Y-%m-%d %H:%M:%S (naive local time)
    daily_file = DAILY_EXPORT.set_axis(DAILY_EXPORT.index.strftime("%Y-%m-%d").rename("date"))
    daily_file.to_csv(EXPORT_PATHS["daily"], date_format=STAMP)
    HOURLY_EXPORT.to_csv(EXPORT_PATHS["hourly"], date_format=STAMP)
    for name, frame in [("daily", DAILY_EXPORT), ("hourly", HOURLY_EXPORT)]:
        print(f"wrote {EXPORT_PATHS[name].relative_to(DATA_DIR.parent)} ({len(frame):,} rows × {frame.shape[1] + 1} columns)")
else:
    print("EXPORT_ENABLED is off: nothing written")

### 6.2 Round-trip check

Both files are read back with a bare `pd.read_csv`: the row counts, three-state flags, the §2.4 structural checks, and every value against the frames in memory.

In [ ]:
def reload(name, frame):
    """A written file, parsed back into the in-memory layout."""
    back = pd.read_csv(EXPORT_PATHS[name])
    index = frame.index.name
    back[index] = pd.to_datetime(back[index], format="%Y-%m-%d" if index == "date" else STAMP)
    back = back.set_index(index)
    for column in back.columns:
        if "_risk_" in column:
            back[column] = back[column].astype("boolean")
        elif "_range_" in column:
            back[column] = pd.to_datetime(back[column], format=STAMP)
    return back


if EXPORT_ENABLED:
    problems, RELOADED = [], {}
    for name, frame, expected_rows in [("daily", DAILY_EXPORT, len(TEST_DAYS)), ("hourly", HOURLY_EXPORT, len(COMMON))]:
        back = reload(name, frame)
        if len(back) != expected_rows:
            problems.append(f"{name}: {len(back):,} rows, expected {expected_rows:,}")
        if list(back.columns) != list(frame.columns):
            problems.append(f"{name}: columns differ from the in-memory frame")
            continue
        for column in back.columns:
            if "_risk_" in column and not back[column].dropna().isin([True, False]).all():
                problems.append(f"{name} {column}: not three-state")
            if pd.api.types.is_float_dtype(frame[column]):
                wrong = int(((back[column] - frame[column]).abs() > 1e-6).sum() + (back[column].isna() != frame[column].isna()).sum())
            else:
                wrong = int(differs(back[column].reindex(frame.index), frame[column]).sum())
            if wrong:
                problems.append(f"{name} {column}: {wrong} values differ after the round trip")
        RELOADED[name] = back
    if not problems:
        problems = structural_problems(RELOADED["daily"], RELOADED["hourly"])
    if problems:
        raise RuntimeError("round-trip check failed:\n  " + "\n  ".join(problems))
    print(
        f"round-trip check passed: {len(RELOADED['daily']):,} test days and {len(RELOADED['hourly']):,} common hours, "
        f"flags three-state, structural checks pass on the files"
    )
else:
    print("EXPORT_ENABLED is off: no files to check")